<div style="background:linear-gradient(135deg,#0a2540 0%,#1a3a5c 60%,#0f3460 100%);
            padding:40px 30px;border-radius:12px;text-align:center;margin-bottom:10px;">
  <h1 style="color:#f4a261;font-size:2em;margin:0 0 8px;">
    🔬 Ciencia de Datos en Descubrimiento de Fármacos
  </h1>
  <h2 style="color:#a8dadc;font-size:1.2em;font-weight:400;margin:0 0 20px;">
    Proyecto final integrador: Pipeline completo de descubrimiento de fármacos asistido por IA
  </h2>
  <p style="color:#cdd6f4;font-size:0.95em;max-width:660px;margin:0 auto;line-height:1.7;">
    Universidad Nacional de Colombia · Extensión UNAL 2026<br>
    <em>Semana 7 — De ChEMBL a candidatos priorizados</em>
  </p>
</div>


---

## ¿Qué es este notebook?

Este es tu **proyecto final del curso**. Aquí aplicarás de forma autónoma
todo el pipeline que aprendiste en las 7 semanas, desde cero, con un target
biológico de tu propia elección.

No es un examen de memoria — es una demostración de que puedes usar
las herramientas del curso para hacer ciencia de datos real en drug discovery.

---

## 🗺️ El pipeline completo

```
        SEMANA 2-3                    SEMANA 3                   SEMANA 4
  ┌─────────────────┐        ┌──────────────────┐       ┌─────────────────────┐
  │  COLECCIÓN Y    │        │  FEATURES Y      │       │  MODELOS QSAR       │
  │  CURACIÓN       │──────► │  ESPACIO QUÍMICO │ ────► │  (Clasificación)    │
  │  (ChEMBL)       │        │  (PCA, t-SNE)    │       │  RF, SVM, XGBoost   │
  └─────────────────┘        └──────────────────┘       └─────────────────────┘
           │                                                        │
           │                    SEMANA 6                            │
           │         ┌──────────────────────────┐                  │
           └────────►│  DOCKING MOLECULAR       │◄─────────────────┘
                     │  Re-docking → batch      │
                     │  ProLIF → scoring        │
                     └──────────────────────────┘
                                  │
                                  ▼
                     ┌──────────────────────────┐
                     │  RANKING FINAL           │
                     │  Top candidatos          │
                     │  Interpretación          │
                     └──────────────────────────┘
```

## 📋 Rúbrica de evaluación

| Sección | Criterio | Puntaje |
|---------|----------|---------|
| **1. Target** | Justificación farmacológica del target elegido | 10 pts |
| **2. Colección** | Criterios de selección documentados; estadísticas del dataset | 10 pts |
| **3. Curación** | Funnel de curación completo; decisiones justificadas | 15 pts |
| **4. Features** | Descriptores calculados; espacio químico visualizado | 10 pts |
| **5. Modelo QSAR** | ≥ 2 modelos comparados; métricas reportadas; Y-scrambling | 20 pts |
| **6. Docking** | Validación RMSD ≤ 2 Å; fingerprints ProLIF reportados | 20 pts |
| **7. Scoring** | Score compuesto calculado; top candidatos identificados | 10 pts |
| **8. Conclusión** | Interpretación farmacológica; limitaciones reconocidas | 5 pts |
| **TOTAL** | | **100 pts** |

## ⏱️ Tiempo estimado por sección

| Sección | Tiempo |
|---------|--------|
| 1–3 (Colección y curación) | 45–60 min |
| 4 (Features) | 20–30 min |
| 5 (Modelo QSAR) | 30–45 min |
| 6–7 (Docking y scoring) | 60–90 min |
| 8 (Conclusión) | 15–20 min |

---

> 💡 **Consejo:** Lee cada sección completa antes de empezar a ejecutar el código.
> Cada celda tiene instrucciones y celdas de reflexión donde debes escribir
> tu interpretación en texto libre.


---
## 0. Instalación y configuración inicial

In [ ]:
# ── Instalar todas las librerías del curso ──────────────────────────────────
!pip install chembl_webresource_client chembl-structure-pipeline rdkit \
             openbabel-wheel meeko vina MDAnalysis prolif spyrmsd \
             scikit-learn xgboost umap-learn tqdm rcsbsearchapi --quiet

print("✅ Librerías instaladas")


In [ ]:
# ── Importaciones ───────────────────────────────────────────────────────────
import os, re, warnings, pickle
from math import log
from collections import defaultdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# RDKit
from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, AllChem, Draw, MACCSkeys
from rdkit.Chem.rdMolDescriptors import GetMorganFingerprintAsBitVect
from rdkit.Chem.Scaffolds import MurckoScaffold
from chembl_webresource_client.new_client import new_client
from chembl_structure_pipeline import standardize_mol

# ML
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, f1_score, matthews_corrcoef,
                             accuracy_score, roc_curve, confusion_matrix,
                             ConfusionMatrixDisplay)
from sklearn.dummy import DummyClassifier
from scipy.spatial import distance

# Docking
try:
    from meeko import MoleculePreparation, PDBQTWriterLegacy
    from openbabel import openbabel
    from vina import Vina
    import MDAnalysis as mda
    import prolif as plf
    from spyrmsd import io, rmsd as spyrmsd_rmsd
    DOCKING_OK = True
    print("✅ Librerías de docking disponibles")
except ImportError as e:
    DOCKING_OK = False
    print(f"⚠️  Algunas librerías de docking no están disponibles: {e}")

try:
    import umap
    UMAP_OK = True
except ImportError:
    UMAP_OK = False

pd.set_option('display.max_columns', 20)
print("✅ Todo listo para el proyecto final")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 🎯 PARÁMETROS DEL PROYECTO — COMPLETA ESTA CELDA
# ══════════════════════════════════════════════════════════════════════

# ── Datos del estudiante ─────────────────────────────────────────────
NOMBRE_ESTUDIANTE  = "Tu nombre aquí"          # ← Completa
FECHA_ENTREGA      = "DD/MM/2026"              # ← Completa

# ── Target biológico ─────────────────────────────────────────────────
CHEMBL_TARGET_ID   = "CHEMBL203"               # ← ChEMBL ID de tu target
PDB_ID             = "1IVO"                    # ← Código PDB (estructura con ligando)
LIGAND_CODE        = "FMM"                     # ← Código del ligando co-cristalizado (3 letras)
CHAIN_ID           = "A"                       # ← Cadena proteica de interés

# SMILES del ligando co-cristalizado (búscalo en PubChem o ChEMBL)
SMILES_LIGANDO_REF = ""                        # ← SMILES del ligando de referencia

# ── Parámetros de curación ───────────────────────────────────────────
UMBRAL_PACTIVIDAD  = 6.0    # pIC50 ≥ 6 = IC50 ≤ 1 µM → activo
MARGEN_GRIS        = 0.5    # ± 0.5 unidades alrededor del umbral → excluir

# ── Carpetas de trabajo ───────────────────────────────────────────────
DIR_ESTRUCTURAS  = "estructuras"
DIR_PDBQT        = "pdbqt"
DIR_MOLS         = "mols"
DIR_DOCKING      = "docking"
DIR_RESULTADOS   = "resultados"

for d in [DIR_ESTRUCTURAS, DIR_PDBQT, DIR_MOLS, DIR_DOCKING, DIR_RESULTADOS]:
    os.makedirs(d, exist_ok=True)

TARGET_SLUG = CHEMBL_TARGET_ID.lower()

print(f"{'='*55}")
print(f"PROYECTO FINAL — {NOMBRE_ESTUDIANTE}")
print(f"{'='*55}")
print(f"  Target ChEMBL:  {CHEMBL_TARGET_ID}")
print(f"  Estructura PDB: {PDB_ID}")
print(f"  Ligando ref.:   {LIGAND_CODE}")
print(f"  Umbral activo:  pActividad ≥ {UMBRAL_PACTIVIDAD} (IC50 ≤ {10**(6-UMBRAL_PACTIVIDAD)*1000:.0f} nM)")


---
## 1. Justificación del target biológico `[10 pts]`

> ✏️ **Completa esta sección en texto libre.**
> Responde las preguntas de abajo en las celdas de texto marcadas con `📝`.


📝 **1.1 ¿Qué es tu target y qué función biológica cumple?**

*[Escribe aquí — 3 a 5 oraciones. Incluye el nombre de la proteína, su familia,
el proceso biológico en el que participa y su papel en la enfermedad.]*


📝 **1.2 ¿Por qué es relevante como target farmacológico?**

*[Escribe aquí — menciona: ¿existe algún fármaco aprobado contra este target?
¿Hay evidencia genética o clínica de que es druggable?
¿Qué enfermedad o condición se busca tratar?]*


📝 **1.3 Justificación de la estructura PDB elegida**

*[Escribe aquí — ¿por qué elegiste esta estructura en particular?
Menciona: resolución, método experimental (cristalografía, cryo-EM, RMN),
presencia del ligando co-cristalizado y relevancia del ligando.]*


In [ ]:
# ── Consultar información del target en ChEMBL ──────────────────────────────
target_client = new_client.target
info_target = target_client.get(CHEMBL_TARGET_ID)

print("INFORMACIÓN DEL TARGET EN ChEMBL")
print("=" * 55)
print(f"  Nombre:       {info_target.get('pref_name','')}")
print(f"  Tipo:         {info_target.get('target_type','')}")
print(f"  Organismo:    {info_target.get('organism','')}")
print()

if info_target.get('target_components'):
    for comp in info_target['target_components']:
        acc = comp.get('accession', '')
        print(f"  UniProt:      {acc}")
        print(f"  → UniProt:    https://www.uniprot.org/uniprot/{acc}")
        print(f"  → PDB:        https://www.rcsb.org/search?q={acc}")
        print(f"  → AlphaFold:  https://alphafold.ebi.ac.uk/entry/{acc}")


In [ ]:
# ── Consultar metadatos del PDB elegido ─────────────────────────────────────
import requests

meta = requests.get(
    f"https://data.rcsb.org/rest/v1/core/entry/{PDB_ID.upper()}",
    timeout=15
).json()

struct = meta.get('struct', {})
exptl  = (meta.get('exptl') or [{}])[0]
rcsb   = meta.get('rcsb_entry_info', {})

print(f"ESTRUCTURA PDB: {PDB_ID.upper()}")
print("=" * 55)
print(f"  Título:       {struct.get('title','')[:65]}")
print(f"  Método:       {exptl.get('method','')}")
res = rcsb.get('resolution_combined', ['N/D'])
print(f"  Resolución:   {res[0] if res else 'N/D'} Å")
fecha = meta.get('rcsb_accession_info',{}).get('initial_release_date','')
print(f"  Depositado:   {fecha[:10]}")
print(f"  Ligandos:     {rcsb.get('nonpolymer_entity_count','N/D')}")
print()
print(f"  → Visualizar: https://www.rcsb.org/structure/{PDB_ID.upper()}")


---
## 2. Colección de datos desde ChEMBL `[10 pts]`


In [ ]:
# ── Funciones de curación (reutilizadas del curso) ──────────────────────────
def read_smiles(smiles):
    if pd.isna(smiles) or str(smiles).strip() == '': return None, "vacío"
    mol = Chem.MolFromSmiles(str(smiles).strip())
    return (mol, None) if mol else (None, "inválido")

def is_valid(mol):
    organic = {'H','B','C','N','O','F','Si','P','S','Cl','Br','I'}
    for a in mol.GetAtoms():
        if a.GetSymbol() not in organic: return False
    return Descriptors.RingCount(mol) >= 1 and Descriptors.NumHAcceptors(mol) >= 1

def select_largest_organic_component(mol):
    sales = {'Cc1ccc(S(=O)(=O)[O-])cc1','O=C(O)C(F)(F)F','O=C(O)C(=O)O'}
    frags = [f for f in Chem.GetMolFrags(mol, asMols=True)
             if is_valid(f) and Chem.MolToSmiles(f) not in sales]
    uniq  = list({Chem.MolToSmiles(f): f for f in frags}.values())
    if not uniq: return None, "sin fragmento válido"
    largest = max(uniq, key=lambda x: x.GetNumHeavyAtoms())
    total   = sum(f.GetNumHeavyAtoms() for f in uniq)
    if largest.GetNumHeavyAtoms() < 0.6 * total: return None, "< 60% átomos"
    return largest, None

def chembl_standardizer(mol):
    try:
        return Chem.MolToSmiles(standardize_mol(mol)), None
    except Exception as e:
        return None, str(e)[:50]

def process_smiles(smiles):
    mol, err = read_smiles(smiles)
    if err: return None, f"paso1: {err}"
    frag, err = select_largest_organic_component(mol)
    if err: return None, f"paso2: {err}"
    smi, err = chembl_standardizer(frag)
    if err: return None, f"paso3: {err}"
    return smi, None

print("✅ Funciones de curación cargadas")


In [ ]:
# ── Descarga desde ChEMBL ───────────────────────────────────────────────────
activity_client = new_client.activity

print(f"Descargando actividades de {CHEMBL_TARGET_ID}...")
actividades = activity_client.filter(
    target_chembl_id=CHEMBL_TARGET_ID,
    standard_units='nM'
).only('canonical_smiles','molecule_chembl_id','pchembl_value',
       'standard_units','standard_value','standard_type')

df_raw = pd.DataFrame.from_records(actividades)
df_raw = df_raw[df_raw['standard_type'].isin(['IC50','Ki','EC50','Kd'])]
df_raw = df_raw.astype({'standard_value': float}).dropna(subset=['standard_value'])
df_raw = df_raw[df_raw['standard_value'] > 0]

print(f"✅ Registros descargados: {len(df_raw)}")
print(f"   Tipos de actividad: {df_raw['standard_type'].value_counts().to_dict()}")
print()
df_raw.head(3)


📝 **2.1 Describe el dataset descargado**

*[Escribe aquí — ¿cuántos registros obtuviste? ¿Qué tipos de actividad dominan?
¿Por qué descargaste solo datos en nM? ¿Hay algo llamativo en la distribución?]*


---
## 3. Curación de datos moleculares `[15 pts]`


In [ ]:
# ── Paso 1: Calcular pActividad ─────────────────────────────────────────────
df_raw['pActividad'] = [-log(v/1e9, 10) for v in df_raw['standard_value']]
df_raw = df_raw[(df_raw['pActividad'] >= 3) & (df_raw['pActividad'] <= 12)]

# ── Paso 2: Estandarizar SMILES ─────────────────────────────────────────────
print("Estandarizando SMILES...")
resultados = [process_smiles(s) for s in tqdm(df_raw['canonical_smiles'])]
df_raw['std_smiles'] = [r[0] for r in resultados]
df_raw['cur_error']  = [r[1] for r in resultados]

n_raw = len(df_raw)
df_std = df_raw.dropna(subset=['std_smiles']).copy()
print(f"  Curación estructural: {len(df_std)}/{n_raw} pasaron ({len(df_std)/n_raw*100:.1f}%)")

# ── Paso 3: Deduplicar ──────────────────────────────────────────────────────
df_std = df_std.sort_values('pActividad', ascending=False)
df_dedup = (df_std.groupby(['std_smiles','standard_type'], as_index=False)
            .agg(molecule_chembl_id=('molecule_chembl_id','first'),
                 IC50_nM=('standard_value','median'),
                 n_mediciones=('standard_value','count'),
                 pActividad=('pActividad','median')))
print(f"  Tras deduplicación: {len(df_dedup)} registros únicos")

# ── Paso 4: Clasificación activo/inactivo ────────────────────────────────────
df_dedup = df_dedup[
    (df_dedup['pActividad'] < UMBRAL_PACTIVIDAD - MARGEN_GRIS) |
    (df_dedup['pActividad'] > UMBRAL_PACTIVIDAD + MARGEN_GRIS)
].copy()
df_dedup['activo'] = (df_dedup['pActividad'] >= UMBRAL_PACTIVIDAD).astype(int)

n_act = df_dedup['activo'].sum()
n_ina = (df_dedup['activo'] == 0).sum()
print(f"  Activos: {n_act} | Inactivos: {n_ina} | Ratio: {n_act/max(n_ina,1):.2f}")


In [ ]:
# ── Funnel de curación ───────────────────────────────────────────────────────
etapas = ['Raw', 'Curación
estructural', 'Dedup +
zona gris', 'Dataset
final']
valores = [n_raw, len(df_std), len(df_dedup) + (len(df_std)-len(df_dedup)),
           len(df_dedup)]
# Corregir a pasos reales
valores = [n_raw, len(df_std), len(df_dedup), len(df_dedup)]

fig, ax = plt.subplots(figsize=(8, 4))
colores = ['#4a90d9','#27ae60','#f39c12','#c084fc']
bars = ax.bar(etapas, valores, color=colores, alpha=0.85,
              edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, valores):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
            str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylabel('Compuestos', fontsize=11)
ax.set_title(f'Funnel de curación — {CHEMBL_TARGET_ID}', fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(f'{DIR_RESULTADOS}/funnel_curacion.png', dpi=130, bbox_inches='tight')
plt.show()

# Guardar dataset curado
archivo_curado = f'{DIR_RESULTADOS}/dataset_{TARGET_SLUG}_curado.csv'
df_dedup.to_csv(archivo_curado, index=False)
print(f"✅ Dataset curado guardado: {archivo_curado}")


📝 **3.1 Justifica las decisiones de curación**

*[Escribe aquí — ¿qué umbral de pActividad elegiste y por qué?
¿Qué porcentaje de moléculas se perdió en cada paso?
¿Hubo algún problema inesperado con los datos de tu target?]*


📝 **3.2 Análisis del balance de clases**

*[Escribe aquí — ¿el dataset está balanceado? Si hay desbalance,
¿cómo afectaría esto a los modelos que entrenes?
¿Qué estrategia usarías para manejarlo?]*


---
## 4. Features y espacio químico `[10 pts]`


In [ ]:
# ── Calcular descriptores y fingerprints ────────────────────────────────────
from rdkit.Chem import QED

DESCRIPTORES = ['MolWt','MolLogP','NumHDonors','NumHAcceptors',
                'TPSA','NumRotatableBonds','RingCount']

def calcular_desc(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return {d: np.nan for d in DESCRIPTORES + ['QED']}
    return {**{d: getattr(Descriptors, d)(mol) for d in DESCRIPTORES},
            'QED': QED.qed(mol)}

def morgan_fp(smiles, radio=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    fp  = GetMorganFingerprintAsBitVect(mol, radio, n_bits)
    arr = np.zeros(n_bits, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

print("Calculando descriptores y fingerprints Morgan...")
descs = pd.DataFrame([calcular_desc(s) for s in tqdm(df_dedup['std_smiles'])])
fps   = [morgan_fp(s) for s in df_dedup['std_smiles']]
mask  = [f is not None for f in fps]
X_fp  = np.array([f for f in fps if f is not None])

df_features = pd.concat([df_dedup.reset_index(drop=True), descs], axis=1)
print(f"✅ Descriptores: {descs.shape} | Fingerprints Morgan: {X_fp.shape}")


In [ ]:
# ── Visualización del espacio químico (PCA sobre fingerprints) ───────────────
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# PCA sobre descriptores
X_desc = descs[DESCRIPTORES].fillna(descs[DESCRIPTORES].median()).values
X_scaled = StandardScaler().fit_transform(X_desc)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# t-SNE sobre fingerprints (submuestra si dataset grande)
from sklearn.manifold import TSNE
N_MAX = min(2000, len(X_fp))
idx_sub = np.random.choice(len(X_fp), N_MAX, replace=False)
X_fp_sub = X_fp[idx_sub]
y_sub = df_dedup['activo'].values[np.array(mask)][idx_sub]

print("Calculando t-SNE...")
tsne = TSNE(n_components=2, perplexity=35, metric='jaccard',
            init='pca', random_state=42, n_iter=800)
X_tsne = tsne.fit_transform(X_fp_sub)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (X2d, titulo) in zip(axes, [(X_pca, 'PCA — Descriptores'), (X_tsne, 't-SNE — Morgan FP')]):
    y_plot = df_dedup['activo'].values[:len(X2d)] if X2d is X_pca else y_sub
    for clase, label, color, alpha in [(0,'Inactivo','#e74c3c',0.35),(1,'Activo','#27ae60',0.65)]:
        m = y_plot == clase
        ax.scatter(X2d[m,0], X2d[m,1], c=color, label=f'{label} (n={m.sum()})',
                   alpha=alpha, s=15, edgecolors='none')
    ax.set_title(titulo, fontsize=11)
    ax.legend(fontsize=9, markerscale=2)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle(f'Espacio químico — {CHEMBL_TARGET_ID}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{DIR_RESULTADOS}/espacio_quimico.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── Estadísticas de los descriptores por clase ───────────────────────────────
print("DESCRIPTORES FISICOQUÍMICOS — ACTIVOS vs INACTIVOS")
print("=" * 60)
for desc in DESCRIPTORES + ['QED']:
    if desc not in df_features.columns: continue
    med_act = df_features[df_features['activo']==1][desc].median()
    med_ina = df_features[df_features['activo']==0][desc].median()
    diff    = med_act - med_ina
    signo   = '▲' if diff > 0 else '▼'
    print(f"  {desc:<22} Activos: {med_act:>7.2f}  Inactivos: {med_ina:>7.2f}  {signo}{abs(diff):.2f}")


📝 **4.1 Análisis del espacio químico**

*[Escribe aquí — ¿Los activos e inactivos están bien separados en el espacio químico?
¿Hay clusters visuales que sugieran familias estructurales distintas?
¿Qué descriptor tiene la mayor diferencia entre activos e inactivos?]*


---
## 5. Modelo QSAR — Clasificación `[20 pts]`

> **Requisito mínimo:** entrenar al menos 2 modelos distintos y comparar sus métricas.
> Reportar: AUC-ROC, F1, MCC, Accuracy. Aplicar Y-scrambling.


In [ ]:
# ── Preparar matrices X e y ──────────────────────────────────────────────────
y = df_dedup['activo'].values
X_desc_clean = descs[DESCRIPTORES].fillna(descs[DESCRIPTORES].median()).values

# División estratificada 80/20
X_tr_d, X_te_d, y_tr, y_te = train_test_split(
    X_desc_clean, y, test_size=0.2, random_state=42, stratify=y)

# Fingerprints (solo las filas válidas)
X_fp_valid  = X_fp
y_fp        = df_dedup['activo'].values[np.array(mask)]
X_tr_fp, X_te_fp, y_tr_fp, y_te_fp = train_test_split(
    X_fp_valid, y_fp, test_size=0.2, random_state=42, stratify=y_fp)

print(f"Train: {len(y_tr)} | Test: {len(y_te)}")
print(f"  % activos train: {y_tr.mean()*100:.1f}%  |  test: {y_te.mean()*100:.1f}%")


In [ ]:
# ── Función de evaluación ────────────────────────────────────────────────────
def evaluar(nombre, y_true, y_pred, y_proba=None):
    return {
        'Modelo':   nombre,
        'AUC-ROC':  round(roc_auc_score(y_true, y_proba), 4) if y_proba is not None else None,
        'F1':       round(f1_score(y_true, y_pred, zero_division=0), 4),
        'MCC':      round(matthews_corrcoef(y_true, y_pred), 4),
        'Accuracy': round(accuracy_score(y_true, y_pred), 4),
    }

resultados_modelos = []

# ── Baseline ─────────────────────────────────────────────────────────────────
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy='most_frequent').fit(X_tr_d, y_tr)
resultados_modelos.append(evaluar('Baseline', y_te, dummy.predict(X_te_d)))

# ── Regresión Logística ──────────────────────────────────────────────────────
pipe_lr = Pipeline([('sc', StandardScaler()),
                    ('m', LogisticRegression(C=1, max_iter=1000,
                                             class_weight='balanced', random_state=42))])
pipe_lr.fit(X_tr_d, y_tr)
resultados_modelos.append(evaluar('Reg. Logística', y_te,
                                   pipe_lr.predict(X_te_d),
                                   pipe_lr.predict_proba(X_te_d)[:,1]))

# ── SVM ──────────────────────────────────────────────────────────────────────
pipe_svm = Pipeline([('sc', StandardScaler()),
                     ('m', SVC(kernel='rbf', C=10, probability=True,
                               class_weight='balanced', random_state=42))])
pipe_svm.fit(X_tr_d, y_tr)
resultados_modelos.append(evaluar('SVM (RBF)', y_te,
                                   pipe_svm.predict(X_te_d),
                                   pipe_svm.predict_proba(X_te_d)[:,1]))

# ── Random Forest (descriptores) ──────────────────────────────────────────────
rf_d = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                               n_jobs=-1, random_state=42)
rf_d.fit(X_tr_d, y_tr)
resultados_modelos.append(evaluar('RF + Descriptores', y_te,
                                   rf_d.predict(X_te_d),
                                   rf_d.predict_proba(X_te_d)[:,1]))

# ── Random Forest (fingerprints Morgan) ──────────────────────────────────────
rf_fp = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                n_jobs=-1, random_state=42)
rf_fp.fit(X_tr_fp, y_tr_fp)
resultados_modelos.append(evaluar('RF + Morgan FP', y_te_fp,
                                   rf_fp.predict(X_te_fp),
                                   rf_fp.predict_proba(X_te_fp)[:,1]))

df_res = pd.DataFrame(resultados_modelos).set_index('Modelo')
print("COMPARACIÓN DE MODELOS")
print("=" * 55)
print(df_res.to_string())


In [ ]:
# ── Curvas ROC ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
modelos_roc = [('Reg. Logística', pipe_lr, X_te_d, y_te),
               ('SVM (RBF)',      pipe_svm, X_te_d, y_te),
               ('RF Descriptores', rf_d,   X_te_d, y_te),
               ('RF Morgan FP',    rf_fp,  X_te_fp, y_te_fp)]
colores_roc = ['#4a90d9','#9b59b6','#27ae60','#f39c12']

for (nombre, modelo, X_te_m, y_te_m), color in zip(modelos_roc, colores_roc):
    proba = modelo.predict_proba(X_te_m)[:,1]
    fpr, tpr, _ = roc_curve(y_te_m, proba)
    auc = roc_auc_score(y_te_m, proba)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{nombre} (AUC={auc:.3f})')

ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5,label='Azar')
ax.set_xlabel('Tasa de Falsos Positivos', fontsize=11)
ax.set_ylabel('Tasa de Verdaderos Positivos', fontsize=11)
ax.set_title(f'Curvas ROC — {CHEMBL_TARGET_ID}', fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{DIR_RESULTADOS}/curvas_roc.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── Y-scrambling (validación) ────────────────────────────────────────────────
# Usar el mejor modelo para validar que aprendió patrones reales

mejor_modelo_nombre = df_res['AUC-ROC'].dropna().idxmax()
print(f"Mejor modelo: {mejor_modelo_nombre}")
print(f"Ejecutando Y-scrambling (20 repeticiones)...")

aucs_scr = []
for _ in range(20):
    y_scr = y_tr.copy(); np.random.shuffle(y_scr)
    m_tmp = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                   n_jobs=-1, random_state=42)
    m_tmp.fit(X_tr_d, y_scr)
    aucs_scr.append(roc_auc_score(y_te, m_tmp.predict_proba(X_te_d)[:,1]))

auc_real = df_res.loc[mejor_modelo_nombre, 'AUC-ROC'] if mejor_modelo_nombre in df_res.index else 0.5

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(aucs_scr, bins=10, color='#e74c3c', alpha=0.7, edgecolor='white')
ax.axvline(auc_real, color='#27ae60', lw=2.5, label=f'Modelo real (AUC={auc_real:.3f})')
ax.axvline(np.mean(aucs_scr), color='#e74c3c', lw=2, ls='--',
           label=f'Scrambling media ({np.mean(aucs_scr):.3f})')
ax.set_xlabel('AUC-ROC'); ax.set_ylabel('Frecuencia')
ax.set_title('Y-scrambling — validación del modelo', fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{DIR_RESULTADOS}/y_scrambling.png', dpi=120, bbox_inches='tight')
plt.show()

diff = auc_real - np.mean(aucs_scr)
print(f"\nDiferencia AUC real vs scrambling: {diff:.4f}")
print("✅ MODELO VÁLIDO" if diff > 0.1 else "⚠️  Diferencia pequeña — revisar")


In [ ]:
# ── Importancia de features ───────────────────────────────────────────────────
imp = pd.Series(rf_d.feature_importances_, index=DESCRIPTORES).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(7, 4))
imp.plot.barh(ax=ax, color='#4a90d9', alpha=0.8, edgecolor='white')
ax.set_xlabel('Importancia (Gini)', fontsize=11)
ax.set_title('Importancia de features — Random Forest', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(f'{DIR_RESULTADOS}/feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Guardar el mejor modelo ───────────────────────────────────────────────────
# Re-entrenar con todos los datos
mejor_modelo_final = RandomForestClassifier(n_estimators=500, class_weight='balanced',
                                            n_jobs=-1, random_state=42)
mejor_modelo_final.fit(X_desc_clean, y)

archivo_modelo = f'{DIR_RESULTADOS}/modelo_qsar_{TARGET_SLUG}.pkl'
with open(archivo_modelo, 'wb') as f:
    pickle.dump({'pipeline': mejor_modelo_final, 'features': DESCRIPTORES,
                 'target': CHEMBL_TARGET_ID, 'metricas': df_res.to_dict()}, f)

df_res.to_csv(f'{DIR_RESULTADOS}/resultados_modelos.csv')
print(f"✅ Modelo guardado: {archivo_modelo}")
print(df_res.to_string())


📝 **5.1 Análisis comparativo de modelos**

*[Escribe aquí — ¿qué modelo obtuvo el mejor AUC-ROC? ¿Y el mejor MCC?
¿Por qué crees que los fingerprints de Morgan funcionan mejor/peor que los descriptores?
¿Qué métrica priorizarías para un problema de drug discovery y por qué?]*


📝 **5.2 Validación por Y-scrambling**

*[Escribe aquí — ¿el modelo superó el scrambling por un margen significativo?
¿Qué significa si la diferencia es pequeña?
¿El Y-scrambling demuestra que el modelo generaliza o solo que no memoriza?]*


📝 **5.3 Interpretación farmacológica del modelo**

*[Escribe aquí — según la importancia de features, ¿qué propiedades fisicoquímicas
distinguen a los activos de los inactivos en tu target?
¿Tiene sentido desde el punto de vista farmacológico?
Por ejemplo: si LogP es el feature más importante, ¿por qué tiene sentido para tu target?]*


---
## 6. Docking molecular y validación del protocolo `[20 pts]`

> Esta sección requiere la estructura PDB y las librerías de docking.
> Si el docking no es posible por restricciones de tiempo/recursos,
> describe el protocolo que seguirías y justifica la elección de parámetros.


In [ ]:
# ── Verificar disponibilidad de librerías de docking ────────────────────────
if not DOCKING_OK:
    print("⚠️  Las librerías de docking no están completamente disponibles.")
    print("   Ejecuta la instalación del inicio y reinicia el entorno.")
else:
    print("✅ Librerías de docking disponibles — procedemos con el protocolo")


In [ ]:
# ── Descargar y preparar la estructura PDB ───────────────────────────────────
if DOCKING_OK:
    import requests

    # Descargar PDB
    r = requests.get(f"https://files.rcsb.org/download/{PDB_ID}.pdb", timeout=30)
    ruta_pdb = f"{DIR_ESTRUCTURAS}/{PDB_ID}.pdb"
    with open(ruta_pdb, 'w') as f:
        f.write(r.text)
    print(f"✅ PDB descargado: {ruta_pdb}")

    # Cargar con MDAnalysis
    u = mda.Universe(ruta_pdb)
    protein = u.select_atoms(f"protein and segid {CHAIN_ID}")
    ligand  = u.select_atoms(f"resname {LIGAND_CODE} and segid {CHAIN_ID}")
    if len(ligand) == 0:
        ligand = u.select_atoms(f"resname {LIGAND_CODE}")

    print(f"  Proteína: {len(protein)} átomos")
    print(f"  Ligando ({LIGAND_CODE}): {len(ligand)} átomos")


In [ ]:
# ── Preparar proteína con pdb2pqr ───────────────────────────────────────────
if DOCKING_OK:
    protein.write(f"{DIR_ESTRUCTURAS}/{PDB_ID}_a.pdb")
    !pdb2pqr \
        --pdb-output={DIR_ESTRUCTURAS}/protein_h.pdb \
        --pH=7.4 --whitespace \
        {DIR_ESTRUCTURAS}/{PDB_ID}_a.pdb \
        {DIR_ESTRUCTURAS}/{PDB_ID}_a.pqr

    u_pqr = mda.Universe(f"{DIR_ESTRUCTURAS}/{PDB_ID}_a.pqr")
    prot_sin_agua = u_pqr.select_atoms("not water")
    prot_sin_agua.atoms.write(f"{DIR_ESTRUCTURAS}/{PDB_ID}_prep.pdb")
    prot_sin_agua.atoms.write(f"{DIR_PDBQT}/{PDB_ID}.pdbqt")

    with open(f"{DIR_PDBQT}/{PDB_ID}.pdbqt", 'r') as f:
        cont = f.read()
    cont = cont.replace('TITLE','REMARK').replace('CRYST1','REMARK')
    with open(f"{DIR_PDBQT}/{PDB_ID}.pdbqt", 'w') as f:
        f.write(cont)

    print(f"✅ Proteína preparada: {DIR_PDBQT}/{PDB_ID}.pdbqt")


In [ ]:
# ── Preparar ligando y calcular caja de docking ──────────────────────────────
if DOCKING_OK:
    import urllib.request

    # Guardar ligando de referencia
    ligand.atoms.write(f"{DIR_ESTRUCTURAS}/{LIGAND_CODE}_org.pdb")

    # Intentar descargar SDF ideal del PDB
    try:
        urllib.request.urlretrieve(
            f"https://files.rcsb.org/ligands/view/{LIGAND_CODE}_ideal.sdf",
            f"{DIR_ESTRUCTURAS}/{LIGAND_CODE}.sdf"
        )
        print(f"✅ SDF del ligando descargado")
    except Exception:
        # Generar desde SMILES
        mol_ref = Chem.MolFromSmiles(SMILES_LIGANDO_REF)
        if mol_ref:
            mol_ref = Chem.AddHs(mol_ref)
            AllChem.EmbedMolecule(mol_ref, AllChem.ETKDGv3())
            w = Chem.SDWriter(f"{DIR_ESTRUCTURAS}/{LIGAND_CODE}.sdf")
            w.write(mol_ref); w.close()
            print("✅ SDF generado desde SMILES")

    # Convertir a PDBQT
    !obabel -i sdf {DIR_ESTRUCTURAS}/{LIGAND_CODE}.sdf -o pdbqt -O {DIR_PDBQT}/{LIGAND_CODE}.pdbqt -p

    # Calcular caja de docking
    pocket_center = ligand.center_of_geometry()
    ligand_box    = ligand.positions.max(axis=0) - ligand.positions.min(axis=0) + 8.0

    print(f"\nCaja de docking:")
    print(f"  Centro: {pocket_center.round(2)}")
    print(f"  Tamaño: {ligand_box.round(2)}")


In [ ]:
# ── Re-docking de validación ─────────────────────────────────────────────────
if DOCKING_OK:
    from openbabel import openbabel

    def pdbqt_to_sdf(pdbqt_string, smiles, output_sdf_path):
        obConv = openbabel.OBConversion()
        obConv.SetInAndOutFormats("pdbqt", "pdb")
        mols, scores = [], []
        for linea in pdbqt_string.split('
'):
            if 'VINA RESULT' in linea:
                try: scores.append(float(linea.split()[3]))
                except: pass
        mol = openbabel.OBMol()
        obConv.ReadString(mol, pdbqt_string)
        mols.append(openbabel.OBMol(mol))
        while obConv.Read(mol): mols.append(openbabel.OBMol(mol))
        smiles_mol = Chem.MolFromSmiles(smiles)
        escritor   = Chem.SDWriter(output_sdf_path)
        for i, ob_mol in enumerate(mols):
            sdf_tmp = obConv.WriteString(ob_mol)
            rdmol   = Chem.MolFromMolBlock(sdf_tmp, removeHs=False, sanitize=False)
            if rdmol is None: continue
            try: rdmol = AllChem.AssignBondOrdersFromTemplate(smiles_mol, rdmol)
            except: pass
            rdmol.SetDoubleProp('vina_score', scores[i] if i<len(scores) else 0.0)
            escritor.write(rdmol)
        escritor.close()

    # Ejecutar re-docking
    print("Ejecutando re-docking...")
    v = Vina(sf_name='vina', verbosity=0)
    v.set_receptor(f"{DIR_PDBQT}/{PDB_ID}.pdbqt")
    v.set_ligand_from_file(f"{DIR_PDBQT}/{LIGAND_CODE}.pdbqt")
    v.compute_vina_maps(center=pocket_center.tolist(), box_size=ligand_box.tolist())
    v.dock(exhaustiveness=16, n_poses=5)
    v.write_poses(f"{DIR_PDBQT}/redocking.pdbqt", n_poses=5, overwrite=True)
    pdbqt_to_sdf(open(f"{DIR_PDBQT}/redocking.pdbqt").read(),
                 SMILES_LIGANDO_REF, f"{DIR_PDBQT}/redocking.sdf")

    energias = v.energies()
    df_dock = pd.DataFrame(energias, columns=['Score','Inter','Intra','Tors','IntraBest'])
    df_dock.index = [f"Pose {i+1}" for i in range(len(df_dock))]
    print(df_dock.round(3).to_string())


In [ ]:
# ── Calcular RMSD con spyrmsd ─────────────────────────────────────────────────
if DOCKING_OK:
    # Guardar ligando org como SDF
    pdb_lig = Chem.MolFromPDBFile(f"{DIR_ESTRUCTURAS}/{LIGAND_CODE}_org.pdb",
                                   removeHs=False, sanitize=False)
    if pdb_lig and SMILES_LIGANDO_REF:
        smol = Chem.MolFromSmiles(SMILES_LIGANDO_REF)
        try:
            lig_bo = AllChem.AssignBondOrdersFromTemplate(smol, pdb_lig)
        except: lig_bo = pdb_lig
        w = Chem.SDWriter(f"{DIR_ESTRUCTURAS}/{LIGAND_CODE}_org.sdf")
        w.write(lig_bo); w.close()

    ref   = io.loadmol(f"{DIR_ESTRUCTURAS}/{LIGAND_CODE}_org.sdf")
    poses = io.loadallmols(f"{DIR_PDBQT}/redocking.sdf")
    ref.strip()

    RMSD = spyrmsd_rmsd.symmrmsd(
        ref.coordinates, [m.coordinates for m in poses],
        ref.atomicnums,  poses[0].atomicnums,
        ref.adjacency_matrix, poses[0].adjacency_matrix,
        minimize=True
    )

    df_dock['RMSD (Å)'] = RMSD
    df_dock['Válida']   = ['✅' if r <= 2.0 else '❌' for r in RMSD]

    print("VALIDACIÓN DEL PROTOCOLO DE DOCKING")
    print("=" * 50)
    print(df_dock[['Score','RMSD (Å)','Válida']].to_string())
    print()
    mejor_rmsd = min(RMSD)
    print(f"Mejor RMSD: {mejor_rmsd:.3f} Å")
    if mejor_rmsd <= 2.0:
        print("✅ PROTOCOLO VÁLIDO — reproduce la pose cristalográfica")
    else:
        print("❌ PROTOCOLO NO VÁLIDO — revisar preparación antes de continuar")

    df_dock.to_csv(f"{DIR_RESULTADOS}/redocking_resultados.csv")


In [ ]:
# ── Fingerprint ProLIF del cristal ───────────────────────────────────────────
if DOCKING_OK:
    rdkit_prot  = Chem.MolFromPDBFile(f"{DIR_ESTRUCTURAS}/{PDB_ID}_prep.pdb",
                                       removeHs=False, sanitize=False)
    protein_mol = plf.Molecule(rdkit_prot)

    lig_mol_cristal = plf.sdf_supplier(f"{DIR_ESTRUCTURAS}/{LIGAND_CODE}_org.sdf")[0]
    fp_cristal = plf.Fingerprint(vicinity_cutoff=8.0, count=True)
    fp_cristal.run_from_iterable([lig_mol_cristal], protein_mol)
    df_cristal = fp_cristal.to_dataframe()

    print("INTERACCIONES DEL LIGANDO CRISTALOGRÁFICO (ProLIF)")
    print("=" * 55)
    df_t = df_cristal.T
    print(df_t[df_t.any(axis=1)].to_string())

    # Visualizar red de interacciones
    fp_cristal.plot_lignetwork(lig_mol_cristal)


📝 **6.1 Análisis del re-docking**

*[Escribe aquí — ¿el protocolo es válido (RMSD ≤ 2 Å)?
Si el RMSD es mayor a 2 Å, ¿qué podrías modificar para mejorar el protocolo?
¿Hay una relación entre el score de Vina y el RMSD de cada pose?]*


📝 **6.2 Interpretación de las interacciones ProLIF**

*[Escribe aquí — ¿qué residuos interactúan con el ligando de referencia?
¿Qué tipos de interacciones dominan (H-bond, hidrofóbicas, π-π)?
¿Algún residuo es el sitio catalítico o es especialmente importante para la función proteica?]*


---
## 7. Ranking final con scoring compuesto `[10 pts]`

> Si el docking en batch no pudo completarse, reporta el ranking
> usando solo el score del modelo QSAR como proxy de actividad.


In [ ]:
# ── Opción A: Scoring compuesto completo (si el docking batch está disponible) ─
# Si tienes el CSV de ranking del NB-DOCK-02, cárgalo aquí:

import glob

archivos_ranking = glob.glob(f'{DIR_DOCKING}/ranking_final_*.csv')
if archivos_ranking:
    df_ranking = pd.read_csv(archivos_ranking[0], index_col=0)
    print(f"✅ Ranking cargado desde docking batch: {archivos_ranking[0]}")
    print(f"   {len(df_ranking)} moléculas rankeadas")
    print()
    print("TOP 10 CANDIDATOS")
    print("=" * 75)
    print(df_ranking.head(10)[['molecule_chembl_id','vina_score',
                                'prolif_sim','score_final','pActividad']].to_string())
else:
    print("⚠️  No se encontró ranking del docking batch.")
    print("   Usando el modelo QSAR como scoring proxy...")


In [ ]:
# ── Opción B: Ranking por predicción del modelo QSAR (fallback) ─────────────
# Usar las probabilidades del mejor modelo como proxy de actividad

y_proba_todos = mejor_modelo_final.predict_proba(X_desc_clean)[:,1]

df_ranking_qsar = df_dedup[['molecule_chembl_id','std_smiles','pActividad']].copy()
df_ranking_qsar['prob_activo_QSAR'] = y_proba_todos
df_ranking_qsar['pred_activo']      = (y_proba_todos >= 0.5).astype(int)
df_ranking_qsar = df_ranking_qsar.sort_values('prob_activo_QSAR', ascending=False).reset_index(drop=True)
df_ranking_qsar.index = range(1, len(df_ranking_qsar)+1)

print("RANKING POR MODELO QSAR (probabilidad de ser activo)")
print("=" * 70)
print(df_ranking_qsar.head(10)[['molecule_chembl_id','pActividad','prob_activo_QSAR']].to_string())

df_ranking_qsar.to_csv(f'{DIR_RESULTADOS}/ranking_qsar_{TARGET_SLUG}.csv')
print(f"\n✅ Ranking QSAR guardado: {DIR_RESULTADOS}/ranking_qsar_{TARGET_SLUG}.csv")


In [ ]:
# ── Visualizar los top candidatos ────────────────────────────────────────────
# Usar el ranking disponible (docking o QSAR)
df_plot = df_ranking_qsar if not archivos_ranking else df_ranking.head(20)
score_col = 'score_final' if 'score_final' in df_plot.columns else 'prob_activo_QSAR'

fig, ax = plt.subplots(figsize=(9, 5))
top20   = df_plot.head(20)
colores = ['#c084fc' if i < 5 else '#7c3aed' if i < 10 else '#4a90d9'
           for i in range(len(top20))]
ax.barh(range(len(top20)), top20[score_col].values[::-1],
        color=colores[::-1], alpha=0.85, edgecolor='white')
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20['molecule_chembl_id'].values[::-1], fontsize=8)
ax.set_xlabel('Score', fontsize=11)
ax.set_title(f'Top 20 candidatos — {CHEMBL_TARGET_ID}', fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
# Leyenda de colores
import matplotlib.patches as mpatches
ax.legend(handles=[
    mpatches.Patch(color='#c084fc', label='Top 5'),
    mpatches.Patch(color='#7c3aed', label='Top 6–10'),
    mpatches.Patch(color='#4a90d9', label='Top 11–20'),
], fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig(f'{DIR_RESULTADOS}/top_candidatos.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── Dibujar las top 5 moléculas ──────────────────────────────────────────────
df_top5 = df_ranking_qsar.head(5) if not archivos_ranking else           df_ranking.head(5).merge(df_dedup[['molecule_chembl_id','std_smiles']], on='molecule_chembl_id')

mols_top5 = [Chem.MolFromSmiles(s) for s in df_top5['std_smiles'] if Chem.MolFromSmiles(s)]
ids_top5  = df_top5['molecule_chembl_id'].tolist()

if mols_top5:
    img = Draw.MolsToGridImage(
        mols_top5[:5],
        molsPerRow=5,
        subImgSize=(300, 250),
        legends=[f"#{i+1}
{mid}" for i, mid in enumerate(ids_top5[:5])]
    )
    display(img)


📝 **7.1 Análisis de los candidatos priorizados**

*[Escribe aquí — ¿qué tienen en común las top 5 moléculas?
¿Pertenecen a la misma familia estructural (scaffold)?
¿Su score experimental (pActividad) es consistente con su ranking computacional?
¿Hay alguna que sea especialmente interesante para síntesis o ensayo experimental?]*


---
## 8. Conclusión e interpretación farmacológica `[5 pts]`


In [ ]:
# ── Reporte integrado final ──────────────────────────────────────────────────
print("=" * 65)
print(f"REPORTE FINAL — PROYECTO: {CHEMBL_TARGET_ID.upper()}")
print(f"Estudiante: {NOMBRE_ESTUDIANTE}")
print("=" * 65)
print()
print(f"TARGET BIOLÓGICO")
print(f"  ChEMBL ID:        {CHEMBL_TARGET_ID}")
print(f"  Estructura PDB:   {PDB_ID}")
print(f"  Ligando ref.:     {LIGAND_CODE}")
print()
print(f"DATASET")
print(f"  Descargados:      {n_raw}")
print(f"  Curados:          {len(df_dedup)}")
print(f"  Activos:          {df_dedup['activo'].sum()} ({df_dedup['activo'].mean()*100:.1f}%)")
print(f"  Inactivos:        {(df_dedup['activo']==0).sum()}")
print()
print(f"MODELO QSAR (mejor)")
mejor = df_res['AUC-ROC'].dropna().idxmax() if df_res['AUC-ROC'].dropna().any() else 'N/D'
if mejor in df_res.index:
    print(f"  Modelo:           {mejor}")
    print(f"  AUC-ROC:          {df_res.loc[mejor,'AUC-ROC']:.4f}")
    print(f"  F1:               {df_res.loc[mejor,'F1']:.4f}")
    print(f"  MCC:              {df_res.loc[mejor,'MCC']:.4f}")
print()
print(f"DOCKING (re-docking)")
if DOCKING_OK and 'df_dock' in dir():
    mejor_rmsd = min(df_dock['RMSD (Å)'])
    print(f"  Mejor RMSD:       {mejor_rmsd:.3f} Å")
    print(f"  Protocolo válido: {'Sí' if mejor_rmsd<=2.0 else 'No'}")
print()
print(f"TOP 3 CANDIDATOS")
for i, (_, row) in enumerate(df_ranking_qsar.head(3).iterrows(), 1):
    print(f"  {i}. {row['molecule_chembl_id']} — pActividad exp: {row['pActividad']:.2f}")


📝 **8.1 Conclusión general**

*[Escribe aquí (mínimo 10 oraciones) —*
- *¿Qué aprendiste sobre tu target a través de este análisis?*
- *¿Los modelos QSAR identificaron propiedades importantes para la actividad?*
- *¿El docking validó o contradijo los resultados del modelo 2D?*
- *¿Cuáles son los candidatos más prometedores y por qué?*
- *¿Qué harías a continuación si tuvieras acceso al laboratorio?]*


📝 **8.2 Limitaciones del análisis**

*[Escribe aquí — Toda ciencia computacional tiene limitaciones.
Menciona al menos 3 limitaciones concretas de tu análisis:*
- *¿Hubo datos de ChEMBL que tuviste que descartar? ¿Por qué?*
- *¿El modelo QSAR puede predecir fuera del dominio de aplicabilidad?*
- *¿El score de Vina es un predictor perfecto de afinidad experimental?*
- *¿Qué experimentos in vitro validarían tus predicciones?]*


📝 **8.3 Reflexión sobre el curso**

*[Escribe aquí — ¿Qué parte del pipeline te pareció más crítica para la calidad
de los resultados finales? ¿Por qué la curación de datos es "el paso más diferenciador"
del curso? ¿Cómo cambiaría el ranking de candidatos si hubieras usado datos mal curados?]*


---
## 9. Archivos entregables


In [ ]:
# ── Verificar y listar todos los archivos generados ─────────────────────────
print("ARCHIVOS GENERADOS POR EL PROYECTO")
print("=" * 60)
print()

archivos_esperados = {
    f'{DIR_RESULTADOS}/funnel_curacion.png':          'Funnel de curación',
    f'{DIR_RESULTADOS}/espacio_quimico.png':          'PCA y t-SNE del espacio químico',
    f'{DIR_RESULTADOS}/curvas_roc.png':               'Curvas ROC de modelos',
    f'{DIR_RESULTADOS}/y_scrambling.png':             'Validación Y-scrambling',
    f'{DIR_RESULTADOS}/feature_importance.png':       'Importancia de features',
    f'{DIR_RESULTADOS}/top_candidatos.png':           'Top candidatos ranked',
    f'{DIR_RESULTADOS}/dataset_{TARGET_SLUG}_curado.csv': 'Dataset curado',
    f'{DIR_RESULTADOS}/resultados_modelos.csv':       'Métricas de modelos',
    f'{DIR_RESULTADOS}/modelo_qsar_{TARGET_SLUG}.pkl': 'Modelo QSAR guardado',
    f'{DIR_RESULTADOS}/ranking_qsar_{TARGET_SLUG}.csv': 'Ranking de candidatos',
    f'{DIR_RESULTADOS}/redocking_resultados.csv':     'Resultados re-docking',
}

todos_ok = True
for ruta, descripcion in archivos_esperados.items():
    existe = os.path.exists(ruta)
    if not existe: todos_ok = False
    estado = '✅' if existe else '❌ FALTA'
    tam = f"({os.path.getsize(ruta)/1024:.0f} KB)" if existe else ''
    print(f"  {estado}  {descripcion:<40} {tam}")

print()
if todos_ok:
    print("✅ TODOS LOS ARCHIVOS ESTÁN PRESENTES — proyecto completo")
else:
    print("⚠️  Algunos archivos faltan — revisa las secciones correspondientes")

print()
print("Para entregar:")
print("  1. Descarga este notebook (.ipynb) con todas las celdas ejecutadas")
print(f"  2. Descarga la carpeta '{DIR_RESULTADOS}/' con todos los archivos")
print("  3. Asegúrate de haber completado TODAS las celdas de texto 📝")


---

## ✅ Lista de verificación final

Antes de entregar, confirma que completaste cada punto:

- [ ] **Sección 0:** Variables de tu target configuradas correctamente
- [ ] **Sección 1:** Tres celdas de texto 📝 completadas con justificación del target
- [ ] **Sección 2:** Dataset descargado y celda 📝 completada
- [ ] **Sección 3:** Funnel de curación generado y dos celdas 📝 completadas
- [ ] **Sección 4:** PCA y t-SNE generados y celda 📝 completada
- [ ] **Sección 5:** ≥ 2 modelos comparados, Y-scrambling aplicado, tres celdas 📝 completadas
- [ ] **Sección 6:** Re-docking ejecutado, RMSD reportado, dos celdas 📝 completadas
- [ ] **Sección 7:** Ranking generado, top 5 moléculas dibujadas, celda 📝 completada
- [ ] **Sección 8:** Tres celdas 📝 completadas (conclusión, limitaciones, reflexión)
- [ ] **Sección 9:** Todos los archivos presentes ✅

---
*NB-PROJECT · Ciencia de Datos en Descubrimiento de Fármacos · UNAL 2026*  
*Pipeline completo: ChEMBL → curación → features → QSAR → docking → ranking*
